# Introduction

---

#  Imports

This section corresponds to the import of all the libraries and modules used in the notebook. Some of these libraries may need to be installed in your Python virtual environment if you haven't done so already.

In [1]:
from branca.colormap import LinearColormap
import osmnx as ox
import networkx as nx
import geopandas as gpd
import pandas as pd
import numpy as np
import momepy
from shapely import geometry
from shapely.geometry import mapping
from shapely.geometry import Polygon, MultiPolygon, LineString, MultiLineString, MultiPoint, Point, box
from shapely.ops import unary_union
from shapely.ops import split
from shapely.ops import nearest_points
from shapely.strtree import STRtree
from scipy.spatial import cKDTree
from sklearn.neighbors import BallTree
import neatnet
import folium
import json
import branca
import math
from geopy.distance import geodesic
import matplotlib.pyplot as plt
from scipy.stats import skew, kurtosis
import contextily as ctx
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from branca.element import MacroElement
from jinja2 import Template
from matplotlib.colors import LinearSegmentedColormap, Normalize, to_hex
from pyproj import CRS
import re
import builtins

---

#  Methodology

This section corresponds to all the methodological steps that were taken to measure street functions and classify them into typologies.
It is divided in three main subsections:
1. Data gathering and processing
2. Calculating street functions
3. Classification of streets into typologies

# 1. Data gathering and processing

## 1.1. OpenStreetMap data retrieving

This subsection describes the process of retrieving all the OpenStreetMap (OSM) data that will be used in the subsequent steps of the methodology.

### 1.1.1. Defining the ``study_area`` parameter and retrieving its polygon

The ``study_area`` parameter corresponds to the polygon that will be used as a bounding limit for all the OpenStreetMap data that will be retrieved. This parameter filters the OSM data and only retrieves features that fall within the specified area. The polygon should be defined in a way that it covers the area of interest for the analysis. Before running the tool, to make sure the analysis is being performed for the desired study area, check [Nominatim](https://nominatim.openstreetmap.org/ui/search.html). Enter in the search bar the name of the area you want to analyze and click on "Search". The map will zoom in to the area you searched for. This can be used to check if the polygon corresponds to the study area. If that is the polygon you wish to analyze, then set the text you've entered in [Nominatim](https://nominatim.openstreetmap.org/ui/search.html) as the parameter in the script. The script will then use this parameter to retrieve the OSM data for the specified area.

In [2]:
study_area = "Município de Lisboa, Portugal"
local_CRS = "epsg:3763"
speed_limits = {
    "motorway": 120,
    "motorway_link": 60,
    "trunk": 100,
    "trunk_link": 60,
    "primary": 50,
    "primary_link": 50,
    "secondary": 50,
    "secondary_link": 50,
    "tertiary": 50,
    "tertiary_link": 50,
    "residential": 50,
    "unclassified": 50,
    "living_street": 20
}

This snippet will use the ``study_area`` parameter to retrieve the polygon that defines the limits of the study area, and it will be used in the results chapter for better communication of the boundaries of the study area. It converts a place name (the study area) into a GeoDataFrame containing the polygon geometry of that place. The Coordinate Reference System (CRS) of the GeoDataFrame is WGS84 (EPSG:4326), so we convert it to EPSG:3763 (a CRS for the portuguese context) for consistency with the rest of the analysis.

In [3]:
study_area_gdf = ox.geocode_to_gdf(study_area)
study_area_gdf = study_area_gdf.to_crs(local_CRS)

### 1.1.2. Retrieving the exclusion mask of the study area

This subsection retrieves the exclusion mask of the study area. The exclusion mask will be later used to (potentially, as it's not perfect) ensure that, when the road network is converted into a street centerlines network, it keeps its integrity and does not overlap buildings and other features that delimit the urban corridors. As it can be seen below, the exclusion mask is created by many OSM layers and filtered for many tags in order to retain the features that will present the best results in the output network. It is recommended that the user modifies the exclusion mask layers and tags depending on the context of the study area, however, the ones that were selected should work as a starting point for most contexts.

Firstly, it is defined a custom function that will be used for filtering out the features that are not at ground level, as they are not relevant for this analysis. The function checks if the feature has a ``layer`` tag and if it is equal to 0. If the feature does not have a ``layer`` tag, it is considered to be at ground level, like most OSM features are tagged. The function returns ``True`` for features that are at ground level and ``False`` for those that are not.

In [4]:
def filter_ground_level(gdf):
    # Ensure 'layer' column exists
    if "layer" not in gdf.columns:
        gdf["layer"] = "0"
    layer_num = pd.to_numeric(gdf["layer"], errors="coerce").fillna(0)
    return gdf[layer_num >= 0]

Then, we define a list of OSM layers and tags that will be used to create the exclusion mask. The layers may include buildings, construction sites, schools, sports pitches, cemeteries, parks, hospitals, among others, depending on how these features were mapped in OSM. Some of these features create noise that make it harder for the simplification algorithm to correctly identify what are actual buildings or areas we might want to keep or just smaller street elements like kiosks or fountains. The tags are used to filter out the features that are not relevant for the exclusion mask.

In [5]:
# Retrieving buildings
buildings = ox.features_from_place(study_area, tags={"building": True})

# Applying filters to the "building" features
if "building" in buildings.columns:
    buildings = buildings[buildings["building"] != "roof"]
    buildings = buildings[buildings["building"] != "container"]
    buildings = buildings[buildings["building"] != "kiosk"]
    buildings = buildings[buildings["building"] != "memorial"]
    buildings = buildings[buildings["building"] != "service"]
    buildings = buildings[buildings["building"] != "guardhouse"]
    buildings = buildings[buildings["building"] != "train_station"]

if "amenity" in buildings.columns:
    buildings = buildings[buildings["amenity"] != "shelter"]
    buildings = buildings[buildings["amenity"] != "fountain"]
    buildings = buildings[buildings["amenity"] != "toilets"]

if "artwork_type" in buildings.columns:
    buildings = buildings[buildings["artwork_type"] != "statue"]

if "historic" in buildings.columns:
    buildings = buildings[buildings["historic"] != "monument"]
    buildings = buildings[buildings["historic"] != "memorial"]

if "memorial" in buildings.columns:
    buildings = buildings[buildings["memorial"] != "statue"]
    buildings = buildings[buildings["memorial"] != "bust"]

if "shop" in buildings.columns:
    buildings = buildings[buildings["shop"] != "kiosk"]

if "bridge:support" in buildings.columns:
    buildings = buildings[buildings["bridge:support"] != "yes"]
    buildings = buildings[buildings["bridge:support"] != "pier"]
    buildings = buildings[buildings["bridge:support"] != "abutment"]
    buildings = buildings[buildings["bridge:support"] != "lift_pier"]
    buildings = buildings[buildings["bridge:support"] != "pivot_pier"]
    buildings = buildings[buildings["bridge:support"] != "pylon"]
    
# Filtering for ground level features
buildings = filter_ground_level(buildings)

In [6]:
# Retrieving construction areas
construction = ox.features_from_place(study_area, tags={"landuse": "construction"})

# Filtering for ground level features
construction = filter_ground_level(construction)

In [7]:
# Retrieving schools
schools = ox.features_from_place(study_area, tags={"amenity": "school"})

# Filtering for ground level features
schools = filter_ground_level(schools)

In [8]:
# Retrieving pitches
pitches = ox.features_from_place(study_area, tags={"leisure": "pitch"})

# Filtering for ground level features
pitches = filter_ground_level(pitches)

In [9]:
# Retrieving cemeteries
cemeteries = ox.features_from_place(study_area, tags={"landuse": "cemetery"})

# Filtering for ground level features
cemeteries = filter_ground_level(cemeteries)

After that, the exclusion mask is prepared for geospatial analysis by first reprojecting the several layers into a common coordinate reference system (EPSG:3763) to ensure spatial consistency. EPSG:3763 is different from EPSG:4326 (WGS84) because it uses meters as units, which is more suitable for distance calculations and spatial operations. It is also the CRS for Portugal, where our case study takes place. Then, all the layers are dissolved into a single geometry to create a unified ``exclusion_mask``. This step is crucial to ensure that the exclusion mask accurately represents all the features that need to be considered when simplifying the road network.

In [10]:
buildings = buildings.to_crs(local_CRS)
construction = construction.to_crs(local_CRS)
schools = schools.to_crs(local_CRS)
pitches = pitches.to_crs(local_CRS)
cemeteries = cemeteries.to_crs(local_CRS)

In [11]:
# Retrieving cemeteries
cemeteries = ox.features_from_place(study_area, tags={"landuse": "cemetery"})

# Filtering for ground level features
cemeteries = filter_ground_level(cemeteries)

After that, the exclusion mask is prepared for geospatial analysis by first reprojecting the several layers into a common coordinate reference system (EPSG:3763) to ensure spatial consistency. EPSG:3763 is different from EPSG:4326 (WGS84) because it uses meters as units, which is more suitable for distance calculations and spatial operations. It is also the CRS for Portugal, where our case study takes place. Then, all the layers are dissolved into a single geometry to create a unified ``exclusion_mask``. This step is crucial to ensure that the exclusion mask accurately represents all the features that need to be considered when simplifying the road network.

In [12]:
buildings = buildings.to_crs(local_CRS)
construction = construction.to_crs(local_CRS)
schools = schools.to_crs(local_CRS)
pitches = pitches.to_crs(local_CRS)
cemeteries = cemeteries.to_crs(local_CRS)

It then extracts and combines the ``geometry`` columns from these layers into a single DataFrame, effectively aggregating all areas that should be protected from simplification. Keeping just the geometry column is important to reduce memory usage and improve performance in subsequent spatial operations.

In [13]:
exclusion_mask = gpd.GeoDataFrame(
    pd.concat([
       buildings[['geometry']],
       construction[['geometry']],
       schools[['geometry']],
       pitches[['geometry']],
       cemeteries[['geometry']]    
    ], ignore_index = True)
)

Finally, it uses a unary union operation to merge these geometries into one cohesive shape, which serves as the exclusion mask. This mask can be used in further spatial operations to preserve important urban features during processes like street network simplification.

In [14]:
exclusion_mask = gpd.GeoSeries(unary_union(exclusion_mask.geometry), crs=local_CRS)

### 1.1.3. Retrieving and cleaning the ``highway`` network of the study area

This subsection retrieves the ``highway`` network of the study area. This network is a representation of the road network in OpenStreetMap, and it includes all types of roads, paths, and other transportation routes.

#### 1.1.3.1. Retrieving and converting to GeoDataFrame

In [15]:
# Filtering for wanted highway types
cf_highway_types = [
    'motorway',
    'motorway_link',
    'trunk',
    'trunk_link',
    'primary',
    'primary_link',
    'secondary',
    'secondary_link',
    'tertiary',
    'tertiary_link',
    'residential',
    'unclassified',
    'living_street',
    'pedestrian'
]

cf = '["highway"~"{}"]'.format('|'.join(cf_highway_types))

# Filtering out area highway types
cf += cf + '["area"!~"yes"]'
cf += cf + '["area:highway"!~"footway"]'
cf += cf + '["area:highway"!~"path"]'
cf += cf + '["area:highway"!~"steps"]'
cf += cf + '["area:highway"!~"pedestrian"]'

# Extend OSMnx useful tags so edges carry directional, PSV, and cycleway info
extra_way_tags = [
    # directional lane counts
    "lanes:forward", "lanes:backward",
    # PSV numeric counts
    "lanes:psv", "lanes:psv:forward", "lanes:psv:backward",
    # PSV per-lane designation strings
    "psv:lanes", "psv:lanes:forward", "psv:lanes:backward",
    "bus:lanes", "bus:lanes:forward", "bus:lanes:backward",  # sometimes used instead of psv
    # tunnels and bridges
    "tunnel", "bridge"
]

ox.settings.useful_tags_way = sorted(set(list(ox.settings.useful_tags_way) + extra_way_tags))

In [16]:
network = ox.graph_from_place(
    study_area,
    custom_filter=cf,
    retain_all=False, 
    simplify=False, 
    truncate_by_edge=True
)

network_gdf = ox.graph_to_gdfs(network, nodes=False, edges=True)

network_gdf = network_gdf[network_gdf.geometry.notnull()]

network_gdf = network_gdf[network_gdf.geometry.type.isin(['LineString', 'MultiLineString'])]

network_gdf = network_gdf.to_crs(local_CRS)

#### 1.1.3.2. Defining "oneway"

The ``oneway`` column defines if a segment is a one-way road or not. Although the direction of traffic is not relevant for this analysis (the "link" function of a street is not dependent on the direction of traffic), it is important to define the ``oneway`` column in order to be able to calculate the potential capacity of the network (since it will define the number of ``lanes`` that will be assumed, namely in streets of lower hierarchy).

In [17]:
# Defining roundabouts as "oneway"=True, if they're empty
network_gdf.loc[network_gdf["junction"].isin(["roundabout"]) & network_gdf["oneway"].isnull(), "oneway"] = True

# Defining "motorway", "motorway_link", "trunk", and "trunk_link" as "oneway"=True, if they're empty
network_gdf.loc[network_gdf["highway"].isin(["motorway", "motorway_link", "trunk", "trunk_link"]) & network_gdf["oneway"].isnull(), "oneway"] = True

# Defining remaining highways as "oneway"= False, if they're empty
network_gdf.loc[network_gdf["oneway"].isnull(), "oneway"] = False

#### 1.1.3.3. Defining "lanes"

The ``lanes`` column defines the number of lanes of each segment. The number of lanes is an important parameter for calculating the potential capacity of the network. The first step is to define the ``lanes`` of lower hierarchical segments (``residential``, ``unclassified`` and ``living_street``). We look for empty rows in the ``lanes`` column for these highway levels. We then define ``1`` lane for one-way roads and ``2`` lanes for two-way roads. This is an assumption, as it is a practice with OSM to have ``oneway`` tagged only for roads that are so.

In [18]:
network_gdf.loc[
    (network_gdf["lanes"].isnull()) & 
    (network_gdf["highway"].isin(["residential", "unclassified", "living_street"])) & 
    (network_gdf["oneway"] == True), 
    "lanes"
] = 1

network_gdf.loc[
    (network_gdf["lanes"].isnull()) & 
    (network_gdf["highway"].isin(["residential", "unclassified", "living_street"])) &
    (network_gdf["oneway"] == False), 
    "lanes"
] = 2

After this, the number of ``lanes`` of the other classes are defined. Luckily, a very substantial part of the network is residential (and that will fall under the previous simplification), and as for the rest of the network, usually the number of ``lanes`` is correctly tagged in OSM. Even so, for this case, the following code sets the number of ``lanes`` base on a weighted average of the length of the segments with that ``highway`` type.

In [19]:
# Calculate weighted average of "lanes" for each "highway" type
# Only use rows where "lanes" is not null and "length" is available
valid_lanes = network_gdf[network_gdf["lanes"].notnull() & network_gdf["length"].notnull()].copy()
valid_lanes["lanes"] = pd.to_numeric(valid_lanes["lanes"], errors="coerce")

weighted_avg_lanes = (
    valid_lanes.groupby("highway")[["lanes", "length"]]
    .apply(lambda df: np.average(df["lanes"], weights=df["length"]))
    .round()
    .astype(int)
)

# Fill missing "lanes" values using the rounded weighted average for each "highway"
def fill_lanes(row):
    if pd.isnull(row["lanes"]):
        return weighted_avg_lanes.get(row["highway"], np.nan)
    return row["lanes"]

network_gdf["lanes"] = network_gdf.apply(fill_lanes, axis=1)

In [20]:
def _to_int(x):
    try:
        if pd.isna(x):
            return np.nan

        # handle lists/tuples: pick first non-null element
        if isinstance(x, (list, tuple)):
            for e in x:
                if not pd.isna(e):
                    x = e
                    break
            else:
                return np.nan

        # strings: try several sane parsing strategies
        if isinstance(x, str):
            s = x.strip()
            if s == "":
                return np.nan
            # normalize separators to pipe
            s = re.sub(r"[;,/]+", "|", s)

            # if pipe-delimited, prefer the first numeric token
            if "|" in s:
                toks = [t.strip() for t in s.split("|") if t.strip() != ""]
                for t in toks:
                    if re.match(r"^[-+]?[0-9]+(\\.[0-9]+)?$", t):
                        return int(float(t))
                # fall through to try parsing first token
                s = toks[0]

            # ranges like 2-3 -> take min(2,3)
            m = re.match(r"^(?P<a>[-+]?[0-9]+(\\.[0-9]+)?)\\s*[-–]\\s*(?P<b>[-+]?[0-9]+(\\.[0-9]+)?)$", s)
            if m:
                a = float(m.group("a"))
                b = float(m.group("b"))
                return int(min(a, b))

            # extract first numeric occurrence (handles "50 km/h", "50mph")
            m = re.search(r"([-+]?[0-9]+(\\.[0-9]+)?)", s)
            if m:
                return int(float(m.group(1)))

            return np.nan

        # numeric types
        return int(float(x))
    except Exception:
        return np.nan

def _count_psv_from_token_string(s):
    if pd.isna(s):
        return 0

    # numeric input (already a count)
    if isinstance(s, (int, float)):
        try:
            if np.isnan(s):
                return 0
            return int(float(s))
        except Exception:
            return 0

    if not isinstance(s, str):
        # try to coerce to number
        try:
            return int(float(s))
        except Exception:
            return 0

    ss = s.strip().lower()
    if ss == "":
        return 0

    # split on common separators
    tokens = [t.strip() for t in re.split(r"[|,;/]+", ss) if t.strip() != ""]
    if not tokens:
        return 0

    # Conservative default: only count tokens that explicitly indicate a dedicated PSV lane.
    psv_explicit = {"designated", "exclusive"}

    count = 0
    for t in tokens:
        # numeric token -> add numeric value
        if re.match(r"^[0-9]+$", t):
            count += int(t)
            continue

        # strip common prefixes
        t_clean = re.sub(r'^(psv:|bus:)', '', t)

        # only accept explicit tokens that unambiguously mark a dedicated PSV lane
        if t_clean in psv_explicit:
            count += 1

    return max(0, int(count))

def _safe_min(a, b):
    try:
        a_missing = pd.isna(a)
        b_missing = pd.isna(b)
        if a_missing and b_missing:
            return np.nan

        a_val = 0 if a_missing else int(float(a))
        b_val = 0 if b_missing else int(float(b))
        return int(max(0, min(a_val, b_val)))
    except Exception:
        return np.nan

In [21]:
# 1) General lane counts
# Ensure total lanes is numeric to avoid string - float errors
lanes_tot = pd.to_numeric(network_gdf.get("lanes"), errors="coerce")

lf = pd.to_numeric(network_gdf.get("lanes:forward"), errors="coerce")
lb = pd.to_numeric(network_gdf.get("lanes:backward"), errors="coerce")

# Start with explicit directional tags if present
lanes_forward_dir = lf.copy()
lanes_backward_dir = lb.copy()

# Where missing, derive from oneway and total lanes
mask_missing_both = lanes_forward_dir.isna() & lanes_backward_dir.isna()
if mask_missing_both.any():
    # oneway -> all lanes in the forward direction
    oneway_mask = mask_missing_both & (network_gdf["oneway"] == True)
    lanes_forward_dir.loc[oneway_mask]  = lanes_tot.loc[oneway_mask]
    lanes_backward_dir.loc[oneway_mask] = 0

    # two-way -> split total lanes
    tw_mask = mask_missing_both & (network_gdf["oneway"] == False)
    tot = lanes_tot.loc[tw_mask].fillna(0).astype(float)
    split_f = np.floor(tot / 2.0).astype(int)
    split_b = (tot - split_f).astype(int)
    lanes_forward_dir.loc[tw_mask]  = split_f
    lanes_backward_dir.loc[tw_mask] = split_b

# Any remaining single-side NaNs: backfill by difference with total
rem_f = lanes_forward_dir.isna() & lanes_tot.notna()
lanes_forward_dir.loc[rem_f] = (lanes_tot.loc[rem_f] - lanes_backward_dir.loc[rem_f].fillna(0)).clip(lower=0)

rem_b = lanes_backward_dir.isna() & lanes_tot.notna()
lanes_backward_dir.loc[rem_b] = (lanes_tot.loc[rem_b] - lanes_forward_dir.loc[rem_b].fillna(0)).clip(lower=0)

# Final integer, nonnegative
lanes_forward_dir  = lanes_forward_dir.fillna(0).astype(int).clip(lower=0)
lanes_backward_dir = lanes_backward_dir.fillna(0).astype(int).clip(lower=0)

In [22]:
# 2) PSV-reserved lanes per direction
# Numeric counts first
psv_tot = pd.to_numeric(network_gdf.get("lanes:psv"), errors="coerce")
# ensure Series (preserve alignment with network_gdf)
psv_tot = pd.Series(psv_tot, index=network_gdf.index)

psv_f   = pd.to_numeric(network_gdf.get("lanes:psv:forward"), errors="coerce")
psv_f   = pd.Series(psv_f, index=network_gdf.index)

psv_b   = pd.to_numeric(network_gdf.get("lanes:psv:backward"), errors="coerce")
psv_b   = pd.Series(psv_b, index=network_gdf.index)

# Token strings (fallbacks)
tok_any = network_gdf.get("psv:lanes")
tok_any = pd.Series(tok_any, index=network_gdf.index)

tok_f   = network_gdf.get("psv:lanes:forward")
tok_f   = pd.Series(tok_f, index=network_gdf.index)

tok_b   = network_gdf.get("psv:lanes:backward")
tok_b   = pd.Series(tok_b, index=network_gdf.index)

# Some data uses bus:* instead of psv:*
tok_any_bus = network_gdf.get("bus:lanes")
tok_any_bus = pd.Series(tok_any_bus, index=network_gdf.index)

tok_f_bus   = network_gdf.get("bus:lanes:forward")
tok_f_bus   = pd.Series(tok_f_bus, index=network_gdf.index)

tok_b_bus   = network_gdf.get("bus:lanes:backward")
tok_b_bus   = pd.Series(tok_b_bus, index=network_gdf.index)

# Start with explicit directional numeric counts
psv_forward_dir  = psv_f.copy()
psv_backward_dir = psv_b.copy()

# If only total numeric count exists, apportion by directional lane share
mask_tot_only = psv_forward_dir.isna() & psv_backward_dir.isna() & psv_tot.notna()
if mask_tot_only.any():
    tot_psv = psv_tot.loc[mask_tot_only].astype(float)
    lf_share = lanes_forward_dir.loc[mask_tot_only].replace(0, np.nan)
    lb_share = lanes_backward_dir.loc[mask_tot_only].replace(0, np.nan)
    denom = (lf_share + lb_share)
    f_alloc = np.floor(tot_psv * (lf_share / denom)).fillna(0)
    b_alloc = (tot_psv - f_alloc).clip(lower=0)
    psv_forward_dir.loc[mask_tot_only]  = f_alloc
    psv_backward_dir.loc[mask_tot_only] = b_alloc

# If still NaN, parse token strings per direction
mask_need_tokens_f = psv_forward_dir.isna()
if mask_need_tokens_f.any():
    src = tok_f.where(tok_f.notna(), tok_any).where(lambda s: s.notna(), tok_any_bus)
    psv_forward_dir.loc[mask_need_tokens_f] = src.loc[mask_need_tokens_f].map(_count_psv_from_token_string)

mask_need_tokens_b = psv_backward_dir.isna()
if mask_need_tokens_b.any():
    src = tok_b.where(tok_b.notna(), tok_any).where(lambda s: s.notna(), tok_b_bus)
    psv_backward_dir.loc[mask_need_tokens_b] = src.loc[mask_need_tokens_b].map(_count_psv_from_token_string)

# Default zeros where still missing
psv_forward_dir  = psv_forward_dir.fillna(0).astype(int)
psv_backward_dir = psv_backward_dir.fillna(0).astype(int)

# Cap PSV counts by available lanes per direction
psv_forward_dir  = np.minimum(psv_forward_dir,  lanes_forward_dir).astype(int)
psv_backward_dir = np.minimum(psv_backward_dir, lanes_backward_dir).astype(int)

In [23]:
# 4) Final outputs per direction
network_gdf["lanes_psv_forward"] = psv_forward_dir.astype(int)
network_gdf["lanes_psv_backward"] = psv_backward_dir.astype(int)
network_gdf["lanes_general_forward"] = (lanes_forward_dir - psv_forward_dir).clip(lower=0).astype(int)
network_gdf["lanes_general_backward"] = (lanes_backward_dir - psv_backward_dir).clip(lower=0).astype(int)

In [24]:
# 5) Forcing all "pedestrian" highways to have no psv or general lanes
ped_mask = network_gdf["highway"] == "pedestrian"
network_gdf.loc[ped_mask, "lanes_psv_forward"] = 0
network_gdf.loc[ped_mask, "lanes_psv_backward"] = 0
network_gdf.loc[ped_mask, "lanes_general_forward"] = 0
network_gdf.loc[ped_mask, "lanes_general_backward"] = 0

In [25]:
# 6) Final outputs without directionality
# For network_gdf
network_gdf["lanes_psv"] = (network_gdf["lanes_psv_forward"] + network_gdf["lanes_psv_backward"]).astype(int)
network_gdf["lanes_general"] = (network_gdf["lanes_general_forward"] + network_gdf["lanes_general_backward"]).astype(int)

Before correcting the ``_status`` == ``new`` segments, we first force "0" values in the ``lanes`` columns on all ``network_gdf``segments that correspond to tunnels or bridges. This is done because these segments do not contribute to the street functions in the same way as other segments, and setting their ``lanes`` to "0" ensures that they are not counted in the aggregations.

In [26]:
# Force tunnels and bridges to have 0 lanes (accept "yes" strings and boolean-like values)
s_tunnel = network_gdf["tunnel"] if "tunnel" in network_gdf.columns else pd.Series([pd.NA] * len(network_gdf), index=network_gdf.index)
s_bridge = network_gdf["bridge"] if "bridge" in network_gdf.columns else pd.Series([pd.NA] * len(network_gdf), index=network_gdf.index)

def bool_like_mask(s):
    # True or 1
    m = s.eq(True) | s.eq(1)
    # string-ish true values: "yes", "true", "1" (case-insensitive)
    m |= s.fillna("").astype(str).str.lower().isin(["yes", "true", "1"])
    return m.fillna(False)

mask_tunnel = bool_like_mask(s_tunnel)
mask_bridge = bool_like_mask(s_bridge)
mask_tb = mask_tunnel | mask_bridge

network_gdf.loc[mask_tb, ["lanes", "lanes_psv", "lanes_general"]] = 0
network_gdf.loc[mask_tb, ["lanes_psv_forward", "lanes_psv_backward",
                          "lanes_general_forward", "lanes_general_backward"]] = 0

#### 1.1.3.4. Defining "maxspeed"

The methodology assumes that the ``maxspeed`` column is defined in km/h. The first step is to assign some fix values to some types of network elements, based on the ``highway`` tag. These values are based on common speed limits for these types of roads, but they may vary depending on the country or region. The values can be adjusted based on local regulations or specific knowledge of the area being analyzed.

In [27]:
# Ensure numeric maxspeed for existing data
network_gdf["maxspeed"] = pd.to_numeric(network_gdf["maxspeed"], errors="coerce")

# Ensure length exists (in meters) for weighted averaging
if "length" not in network_gdf.columns:
    network_gdf["length"] = network_gdf.geometry.length.astype(float)

# 1) Keep existing maxspeed as-is (already numeric)
# 2) Fill missing from user-provided dictionary
mask_missing = network_gdf["maxspeed"].isnull()
if mask_missing.any():
    mapped = network_gdf.loc[mask_missing, "highway"].map(speed_limits)
    mapped = pd.to_numeric(mapped, errors="coerce")
    network_gdf.loc[mask_missing, "maxspeed"] = mapped

# 3) Fallback: mode by highway type
# Compute mode of "maxspeed" for each "highway" type
valid_speed = network_gdf[network_gdf["maxspeed"].notnull()].copy()
valid_speed["maxspeed"] = pd.to_numeric(valid_speed["maxspeed"], errors="coerce")

mode_maxspeed = (
    valid_speed.groupby("highway")["maxspeed"]
    .agg(lambda x: x.mode().iloc[0] if not x.mode().empty else np.nan)
)

# Fill missing "maxspeed" values using the mode for each "highway"
def fill_maxspeed(row):
    if pd.isnull(row["maxspeed"]):
        return mode_maxspeed.get(row["highway"], np.nan)
    return row["maxspeed"]

network_gdf["maxspeed"] = network_gdf.apply(fill_maxspeed, axis=1)
# convert to numeric (ints) where possible
network_gdf["maxspeed"] = pd.to_numeric(network_gdf["maxspeed"], errors="coerce").astype("Float64")

## 1.2. Deriving a street centerlines network

This subsection corresponds to the creation of a street centerlines network from the ``highway`` network retrieved from OSM in the previous subsection. The street centerlines network is a simplified representation of urban corridors, represented by a line that runs through the center of the street (hence the name) and will be the base on which the street functions will be calculated.

### 1.2.1. Preparing the network for simplification

Since some of the geometries in the road network will be merged to street centerlines, namely in streets that present multiple carriageways, some of the original network information will be lost. With this being said, the next step removes all columns from the GeoDataFrame except for the ``geometry`` column (where the spatial information is kept) and other columns that retain information related to the corridor, such as ``name`` and ``osmid``. This is done to ensure that only the geometries are passed to the ``neatnet.neatify()`` function, which is responsible for deriving the street centerlines.

In [28]:
_link_map = {
    "motorway_link": "motorway",
    "trunk_link": "trunk",
    "primary_link": "primary",
    "secondary_link": "secondary",
    "tertiary_link": "tertiary"
}

network_gdf["highway_class"] = network_gdf["highway"].map(_link_map).fillna(network_gdf["highway"])

In [29]:
highway_priority = [
    "motorway",
    "trunk",
    "primary",
    "secondary",
    "tertiary",
    "residential",
    "unclassified",
    "living_street",
    "pedestrian"
]

### 1.2.2. Simplifying the network

The ``neatnet.neatify()`` function is then called to derive the street centerlines from the road network. This function simplifies the road network by removing unnecessary details while preserving the overall structure and connectivity of the streets. The result is a simplified representation of the street network, which will be used for the analysis. This function inputs as parameters the network GeoDataFrame only with the ``geometry``, ``name`` and ``osmid`` columns and projected in EPSG:3763, and the exclusion mask that was set earlier. The output is a GeoDataFrame containing the street centerlines (``street_lines``).

In [30]:
street_lines = neatnet.neatify(network_gdf, exclusion_mask = exclusion_mask.geometry,)

c:\Users\Asus\anaconda3\envs\ox\Lib\site-packages\neatnet\geometry.py:269: UserWarning: Could not create a connection as it would lead outside of the artifact.
  additions, splits = snap_to_targets(
c:\Users\Asus\anaconda3\envs\ox\Lib\site-packages\neatnet\geometry.py:269: UserWarning: Could not create a connection as it would lead outside of the artifact.
  additions, splits = snap_to_targets(
c:\Users\Asus\anaconda3\envs\ox\Lib\site-packages\neatnet\nodes.py:52: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  cleaned_roads = pd.concat(
c:\Users\Asus\anaconda3\envs\ox\Lib\site-packages\neatnet\simplify.py:535: UserWarning: Could not create a connection as it would lead outside of the artifact.
  nx_gx_cluster(
c:\Users\Asus\anaconda3\envs\ox

### 1.2.3. Correcting the ``_status`` == ``new`` segments of the output network
During the process of running ``neatnet.neatify()``, the tool is not able to keep some of the data from the original network, namely the new segments whose ``_status`` column is set to ``new``. This happens because, as hinted earlier, these segments result from a merge of multiple original segments during the simplification process and do not have a direct correspondence to any segment of that original network. As a result, they do not inherit any attributes from the original network, and their attribute values are null. To fix this, the following code "probes" the original network to find the nearest segment to each of the new segments. It then copies the attributes from the nearest segment in the original network to the new segment in the simplified network. This way, the new segments will have meaningful attribute values that are consistent with the original network.

#### 1.2.3.1. ID-ing segments and generating probes

Before starting the probing methodology, we give both the original carriageways (``network_gdf_essential``) and the new streets (``street_lines``) unique identifiers. These IDs (``orig_id`` and ``street_id``) will help us keep track of matches later when exploding geometries or merging attributes. Resetting indices ensures the IDs are consistent and reproducible.

In [31]:
def assign_ids(gdf, col_name):
    gdf = gdf.reset_index(drop=True).copy()
    gdf[col_name] = gdf.index.astype(str)
    return gdf

network_gdf = assign_ids(network_gdf, 'orig_id')
street_lines = assign_ids(street_lines, 'street_id')

New segments are extracted from the ``street_lines`` GeoDataFrame in order to apply the procedure to just the segments that require and save computing time.

In [32]:
street_lines_new = street_lines.loc[street_lines['_status'] == 'new', ['street_id', '_status', 'geometry']].copy()
street_lines_new = gpd.GeoDataFrame(street_lines_new, crs=street_lines.crs)

Points are then interpolated every 10 meters, and at each point a perpendicular “probe” is drawn. These probes cut across the full width of the street, ensuring that even very wide boulevards intersect with the relevant original carriageways. This approach keeps the benefits of splitting (granularity for attribute assignment) while allowing probes to be generated smoothly along the entire street.

In [33]:
probe_half_length = 40   # 40 meters to each side (total 80-meter probe)
probe_spacing     = 10   # probe every 10 meters along the line

# dissolve to one geometry per street_id
dissolved = street_lines_new.dissolve(by='street_id', as_index=False)

probe_recs = []
for _, row in dissolved.iterrows():
    sid  = row.street_id
    geom = row.geometry
    branches = geom.geoms if geom.geom_type == 'MultiLineString' else [geom]
    for seg in branches:
        seg_len = seg.length
        if seg_len <= probe_spacing:
            continue
        dists = np.arange(probe_spacing, seg_len, probe_spacing)

        # unit perpendicular from endpoints
        p0, p1 = Point(seg.coords[0]), Point(seg.coords[-1])
        dx, dy = p1.x - p0.x, p1.y - p0.y
        ux, uy = -dy, dx
        norm   = (ux**2 + uy**2)**0.5
        if norm == 0:
            continue
        ux, uy = ux/norm, uy/norm

        for d in dists:
            mid  = seg.interpolate(d)
            end1 = Point(mid.x + ux*probe_half_length, mid.y + uy*probe_half_length)
            end2 = Point(mid.x - ux*probe_half_length, mid.y - uy*probe_half_length)
            probe_recs.append({'street_id': sid, 'geometry': LineString([end2, end1])})

probes = gpd.GeoDataFrame(probe_recs, crs=street_lines_new.crs)
probes['probe_seq'] = probes.groupby('street_id').cumcount()
probes['probe_id']  = probes['street_id'] + '_' + probes['probe_seq'].astype(str)
probes = probes.drop(columns='probe_seq')

Once the ``probes`` have been generated, the next step is to determine which original carriageway segments they intersect. This is done efficiently by first using a spatial index to identify candidate lines that fall within the bounding box of each probe, which greatly reduces the number of geometries that need to be checked. Only the candidates that truly intersect the probe are retained. This process assigns to every probe a list of one or more ``orig_id`` values corresponding to the carriageways that it touches. In practice, this means that each probe effectively “samples” the original dataset, recording which existing lines are encountered when extending across the width of the new street. The result is a set of probe geometries, each linked to the original lines it intersects, which provides the foundation for inferring attributes.

In [34]:
probes = probes.to_crs(network_gdf.crs)
sidx   = network_gdf.sindex

orig_ids_list = []
for g in probes.geometry:
    cand_idx = list(sidx.intersection(g.bounds))
    if not cand_idx:
        orig_ids_list.append([])
        continue
    cand = network_gdf.iloc[cand_idx]
    hits = cand[cand.intersects(g)]
    orig_ids_list.append(hits['orig_id'].tolist())

probes['orig_ids'] = orig_ids_list

Because a single probe can intersect multiple original carriageways, the results from the previous step are stored as lists of identifiers. To analyze these in a structured way, the data is “exploded”, meaning that each probe–carriageway combination is written out as its own row in the dataset. This ensures that every intersected line is represented explicitly rather than hidden inside a list. Once exploded, the attributes from the original dataset, such as the ``name`` of the street and its ``osmid`` identifier, can be joined directly to each probe–carriageway combination. At this point, the dataset expresses exactly which probe touched which original line, along with the relevant attributes, making it possible to analyze the probes individually before moving on to aggregation.

In [35]:
exploded = (
    probes[['probe_id','street_id','geometry','orig_ids']]
    .explode('orig_ids')
    .dropna(subset=['orig_ids'])
    .rename(columns={'orig_ids':'orig_id'})
)

merged = exploded.merge(
    network_gdf.drop(columns='geometry'),
    on='orig_id',
    how='left'
)

#### 1.2.3.2. Probe-level aggregations

Each probe now has potentially several attribute candidates, since it may have intersected more than one original line. To resolve these, a probe-level aggregation is performed (later, we perform a street-level aggregation).

For the ``name`` attribute, the statistical mode (the most frequently occurring value) is selected. If no single name dominates and there is a tie, the tied values are preserved as a list so that no information is lost. For ``highway``, this is done similarly, with ties being preserved as a list as well.

For ``osmid``, which can naturally represent multiple carriageways in parallel, a list of all unique identifiers encountered by the probe is retained. 

For ``lanes``, ``lanes_general`` and ``lanes_psv``, the sum is taken across all intersected lines, as the total number of lanes will contribute to the street's overall capacity.

For ``maxspeed``, the mode is again taken and ties are resolved by checking the highest value.

This produces a clean, consolidated record for each probe: one geometry that carries either a single attribute value or a structured collection of possible values. By summarizing at the probe level, the dataset becomes easier to interpret and ready for the next stage of aggregation at the street scale.

In [36]:
# Defining mode or list custom function (for "name")
def mode_or_list(series):
    if series.empty:
        return np.nan
    mode_series = series.mode()
    if not mode_series.empty:
        return mode_series.iloc[0]
    else:
        return series.tolist()

In [37]:
# Defining mode or max custom function (for "maxspeed")
def mode_or_max(series):
    if series.nunique() == 1:
        return series.iloc[0]
    else:
        return series.mode().iloc[0] if not series.mode().empty else series.max()

In [38]:
# Defining list custom function (for "osmid")
def list_unique_flat(series):
    vals = []
    for v in series.dropna():
        if isinstance(v, (builtins.list, builtins.tuple, builtins.set)):
            vals.extend(v)
        else:
            vals.append(v)
    seen = set()
    out = []
    for item in vals:
        if item not in seen:
            seen.add(item)
            out.append(item)
    return out

In [39]:
# Aggregating to the probes from the orig_id (using the custom functions, depending on the column)
# We're still on probe-level aggregation here
aggregated = merged.groupby(['probe_id', 'street_id', 'geometry']).agg({
    'name': mode_or_list,
    'osmid': list_unique_flat,
    'highway': mode_or_list,
    'maxspeed': mode_or_max,
    'lanes_psv': 'sum',
    'lanes_general': 'sum',
}).reset_index()

#### 1.2.3.3. Street-level aggregations

Probe-level results are combined by ``street_id`` to assign attributes to each new street in a way that is both lightweight and deterministic. 

For ``name``, we do the same as in the probe-level aggregation: we take the mode across all probes belonging to the street. If there is a tie for most frequent, the tied values are preserved as a list so that no information is lost.

For ``highway``, the mode is used to find the most common classification among the probes, with ties prioritizing the ``highway_priority`` ranking to select the type of higher hierarchical level. 

For ``osmid``, the union of all identifiers observed across the street’s probes is kept by design, since multiple parallel carriageways may legitimately belong to the same street; no further tie-breaking is needed.

For ``lanes``, ``lanes_general`` and ``lanes_psv``, since this is the street-level aggregation, we cannot sum the values. This time, we want the mode (across all probes belonging to the street) to represent the most common configuration of lanes along the street. If there is a tie for most frequent, we take the lowest value among the tied candidates, under the assumption that streets always have the minimum number of lanes across their length and that the street's capacity is strongly conditioned by its narrowest point.

In [40]:
# Defining the mode or highway_priority hierarchy custom function (for "highway_class")
def mode_or_hierarchy(series):
    if series.empty:
        return np.nan
    mode_series = series.mode()
    if not mode_series.empty:
        return mode_series.iloc[0]
    else:
        return series.tolist()

In [41]:
# Defining mode or min custom function (for "lanes", "lanes_psv", "lanes_general", "lanes_cycle")
def mode_or_min(series):
    if series.empty:
        return None
    mode = series.mode()
    if not mode.empty:
        return mode[0]
    return series.min()

In [42]:
# Aggregating to the streets through the probes (with a "maxspeed" adjustment for user-provided limits)

# 1) aggregate with a named maxspeed from data
street_lines_new = aggregated.groupby('street_id').agg(
    name=('name', mode_or_list),
    osmid=('osmid', list_unique_flat),
    highway=('highway', mode_or_hierarchy),
    maxspeed_mode=('maxspeed', mode_or_max),
    lanes_psv=('lanes_psv', mode_or_min),
    lanes_general=('lanes_general', mode_or_min),
).reset_index()

# 2) override with user-provided limits, fallback to mode_or_max
#    priority: speed_limits[highway] -> maxspeed_mode
street_lines_new['maxspeed_user'] = street_lines_new['highway'].map(speed_limits)
street_lines_new['maxspeed'] = street_lines_new['maxspeed_user'].fillna(street_lines_new['maxspeed_mode'])

# clean up and ensure numeric
street_lines_new.drop(columns=['maxspeed_user', 'maxspeed_mode'], inplace=True)
street_lines_new['maxspeed'] = pd.to_numeric(street_lines_new['maxspeed'], errors='coerce').astype('Float64')

C:\Users\Asus\AppData\Local\Temp\ipykernel_27188\2704750973.py:5: UserWarning: Unable to sort modes: '<' not supported between instances of 'str' and 'list'
  mode_series = series.mode()
C:\Users\Asus\AppData\Local\Temp\ipykernel_27188\2704750973.py:5: UserWarning: Unable to sort modes: '<' not supported between instances of 'list' and 'str'
  mode_series = series.mode()


In [43]:
# columns produced by the probe pipeline
cols = ['name','osmid','highway','maxspeed','lanes_psv','lanes_general']

# attach new attributes next to the full network
aug = street_lines.merge(
    street_lines_new[['street_id'] + cols].add_suffix('_new').rename(columns={'street_id_new':'street_id'}),
    on='street_id',
    how='left'
)

# replace attributes only where _status == 'new'
mask = aug['_status'].eq('new')
for c in cols:
    aug[c] = np.where(mask, aug[f'{c}_new'], aug[c])

# clean up
aug = aug.drop(columns=[f'{c}_new' for c in cols])

# optional: ensure maxspeed dtype
aug['maxspeed'] = pd.to_numeric(aug['maxspeed'], errors='coerce').astype('Float64')

# result: full network with “new” rows updated
street_lines = aug

# 2. Calculating street functions

## 2.1. Calculating the "link" function

### 2.1.1. Calculating the potential capacity of each street segment (people/hour)

In [44]:
# Defining general lane base capacity per "highway" type,between 600 and 1600 p/h/lane
general_lane_base_capacity = {
    "motorway": 1600,
    "motorway_link": 1200,
    "trunk": 1400,
    "trunk_link": 1000,
    "primary": 1000,
    "primary_link": 800,
    "secondary": 800,
    "secondary_link": 600,
    "tertiary": 600,
    "tertiary_link": 600,
    "unclassified": 600,
    "residential": 600,
    "living_street": 300,
    "pedestrian": 0
}

# Defining PSV lane base capacity per "highway" type, between 1000 and 2800 p/h/lane
psv_lane_base_capacity = {
    "motorway": 2800,
    "motorway_link": 2000,
    "trunk": 2400,
    "trunk_link": 1800,
    "primary": 1800,
    "primary_link": 1500,
    "secondary": 1500,
    "secondary_link": 1200,
    "tertiary": 1200,
    "tertiary_link": 1000,
    "unclassified": 1000,
    "residential": 1000,
    "living_street": 500,
    "pedestrian": 0
}

In [45]:
# Calculating potential capacities
street_lines["pot_capacity"] = street_lines["lanes_general"] * street_lines["highway"].map(general_lane_base_capacity).fillna(0) + \
                               street_lines["lanes_psv"] * street_lines["highway"].map(psv_lane_base_capacity).fillna(0)

### 2.1.2. Calculating edge betweenness centrality (EBC)

#### 2.1.2.1. Creating a NetworkX graph from the street centerlines

In [ ]:
def endpoints(geom):
    if isinstance(geom, LineString):
        c = geom.coords
        return [((c[0][0], c[0][1]), (c[-1][0], c[-1][1]))]
    elif isinstance(geom, MultiLineString):
        pairs = []
        for part in geom.geoms:
            c = part.coords
            pairs.append(((c[0][0], c[0][1]), (c[-1][0], c[-1][1])))
        return pairs
    else:
        return []

In [ ]:
street_lines_MultiGraph = nx.Graph()
for sid, geom, L in zip(street_lines["street_id"], street_lines.geometry, street_lines["length"]):
    for a, b in endpoints(geom):   # iterate through all pairs
        street_lines_MultiGraph.add_edge(a, b, street_id=sid, length=float(L))

#### 2.1.2.2. Calculating EBC values for the abstract graph

In [ ]:
edge_bc = nx.edge_betweenness_centrality(street_lines_MultiGraph, 
                                         weight = "length",
                                         normalized = "True"
                                         )

#### 2.1.2.3. Mapping EBC values back to the street centerlines GeoDataFrame

In [ ]:
id_to_bc = {}

for (u, v, data) in street_lines_MultiGraph.edges(data=True):
    sid = data["street_id"]
    # For an undirected Graph, edge key is (u,v) or (v,u) — use either
    id_to_bc[sid] = edge_bc.get((u, v), edge_bc.get((v, u)))

street_lines["bet_centrality"] = street_lines["street_id"].map(id_to_bc)

### 2.1.3. Calculating the "link" function

# X. MAPS

In [ ]:
# Folium map of street_lines with cols as tooltips (field names, no aliases)
# plus network_gdf and probes layers

# convert to WGS84 for display
sl_wgs = street_lines.to_crs(epsg=4326).copy()
ng_wgs = network_gdf.to_crs(epsg=4326).copy()
probes_wgs = probes.to_crs(epsg=4326).copy()

# ensure tooltipable values are strings (handle lists/dicts/arraylike) robustly
def _to_tooltip_val(v):
    if v is None:
        return ""
    try:
        is_na = pd.isna(v)
        if isinstance(is_na, (np.ndarray, pd.Series, list, tuple)):
            if all(pd.isna(x) for x in v):
                return ""
        elif is_na:
            return ""
    except Exception:
        pass
    if isinstance(v, dict):
        return json.dumps(v, ensure_ascii=False)
    if isinstance(v, (list, tuple, set, np.ndarray, pd.Series)):
        try:
            seq = list(v)
        except Exception:
            return str(v)
        return json.dumps(seq, ensure_ascii=False)
    return str(v)

# prepare tooltip fields per layer (only include existing columns)
street_tooltip_fields = [c for c in cols if c in sl_wgs.columns]
network_tooltip_fields = [c for c in ['name','osmid','highway','maxspeed','lanes','lanes_general','lanes_psv'] if c in ng_wgs.columns]
probes_tooltip_fields = [c for c in ['probe_id','street_id','orig_ids'] if c in probes_wgs.columns]

# stringify tooltip fields
for c in street_tooltip_fields:
    sl_wgs[c] = sl_wgs[c].apply(_to_tooltip_val)
for c in network_tooltip_fields:
    ng_wgs[c] = ng_wgs[c].apply(_to_tooltip_val)
for c in probes_tooltip_fields:
    probes_wgs[c] = probes_wgs[c].apply(_to_tooltip_val)

# compute map center
center_pt = sl_wgs.geometry.unary_union.centroid
m = folium.Map(location=[center_pt.y, center_pt.x], tiles="cartodbpositron", zoom_start=13)

# street_lines layer (blue)
tooltip_sl = folium.features.GeoJsonTooltip(fields=street_tooltip_fields, aliases=None, localize=True)
folium.GeoJson(
    sl_wgs[street_tooltip_fields + ["geometry"]].to_json(),
    name="street_lines",
    tooltip=tooltip_sl,
    style_function=lambda feat: {"color": "#3186cc", "weight": 2, "opacity": 0.8},
).add_to(m)

# network_gdf layer (dark gray, thin)
if len(network_tooltip_fields):
    tooltip_ng = folium.features.GeoJsonTooltip(fields=network_tooltip_fields, aliases=None, localize=True)
else:
    tooltip_ng = None

folium.GeoJson(
    ng_wgs[network_tooltip_fields + ["geometry"]].to_json(),
    name="network_gdf",
    tooltip=tooltip_ng,
    style_function=lambda feat: {"color": "#444444", "weight": 1, "opacity": 0.7},
).add_to(m)

# probes layer (red)
if len(probes_tooltip_fields):
    tooltip_probes = folium.features.GeoJsonTooltip(fields=probes_tooltip_fields, aliases=None, localize=True)
else:
    tooltip_probes = None

folium.GeoJson(
    probes_wgs[probes_tooltip_fields + ["geometry"]].to_json(),
    name="probes",
    tooltip=tooltip_probes,
    style_function=lambda feat: {"color": "#d62728", "weight": 1.5, "opacity": 0.9},
).add_to(m)

folium.LayerControl().add_to(m)

# Save as HTML
m.save("street_lines_inspection.html")

In [54]:
# Folium map showing pot_capacity symbology (color only, fixed width)
sl = sl_wgs.copy()
# ensure numeric pot_capacity
sl['pot_capacity'] = pd.to_numeric(sl.get('pot_capacity'), errors='coerce').fillna(0.0)

vmin = float(sl['pot_capacity'].replace(0, np.nan).min()) if not sl['pot_capacity'].replace(0, np.nan).empty else 0.0
vmax = float(sl['pot_capacity'].max())
if np.isnan(vmin):
    vmin = 0.0
if vmax <= 0:
    vmax = max(1.0, vmin)

# color ramp (low -> high)
cmap = LinearColormap(['#ffffbf', '#d7191c'], vmin=vmin, vmax=vmax)
cmap.caption = "Potential capacity (people / hour)"

# map center
center = sl.geometry.unary_union.centroid
m = folium.Map(location=[center.y, center.x], tiles="cartodbpositron", zoom_start=13)

# style function using pot_capacity (safe numeric conversion) — fixed line width
def _style_function(feat):
    try:
        val = float(feat['properties'].get('pot_capacity') or 0)
    except Exception:
        val = 0.0
    color = cmap(val) if val > 0 else "#cccccc"
    weight = 2  # fixed line width
    return {"color": color, "weight": weight, "opacity": 0.9}

# include the previously-defined "cols" fields as tooltips in addition to pot_capacity
tooltip_fields = ['_status'] + ['pot_capacity'] + [c for c in cols if c in sl.columns]

# stringify tooltip fields for robustness (uses _to_tooltip_val defined earlier)
for c in tooltip_fields:
    sl[c] = sl[c].apply(_to_tooltip_val)

tooltip = folium.features.GeoJsonTooltip(fields=tooltip_fields, aliases=None, localize=True)

folium.GeoJson(
    sl[tooltip_fields + ["geometry"]].to_crs(epsg=4326).to_json(),
    name="pot_capacity",
    style_function=_style_function,
    tooltip=tooltip
).add_to(m)

cmap.add_to(m)
folium.LayerControl().add_to(m)

# display in notebook and save as html for inspection
m.save("street_lines_pot_capacity.html")

C:\Users\Asus\AppData\Local\Temp\ipykernel_27188\4001208612.py:18: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  center = sl.geometry.unary_union.centroid
